In [7]:
import numpy as np
import pandas as pd

df = pd.read_csv('./files/clustered_books.csv')

In [8]:
df.head()

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time (minutes),Genre,Rank,combined_text,cleaned_text,cluster
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313,10080.0,"Over the past three years, Jay Shetty has beco...",654.0,Personal Success,1,Think Like a Monk: The Secret of How to Harnes...,think like monk secret harness power positivit...,3
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658,615.0,Brought to you by Penguin.,203.0,Meditation,1,Ikigai: The Japanese Secret to a Long and Happ...,ikigai japanese secret long happy life brought...,0
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174,10378.0,"In this generation-defining self-help guide, a...",317.0,Personal Success,3,The Subtle Art of Not Giving a F*ck: A Counter...,subtle art giving fck counterintuitive approac...,3
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614,888.0,Brought to you by Penguin.,335.0,Psychology,1,Atomic Habits: An Easy and Proven Way to Build...,atomic habit easy proven way build good habit ...,0
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302,1005.0,"Stop going through life, Start growing throug...",385.0,Literary Essays,1,Life's Amazing Secrets: How to Find Balance an...,life amazing secret find balance purpose life ...,3


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3025 entries, 0 to 3024
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Book Name                 3025 non-null   object 
 1   Author                    3025 non-null   object 
 2   Rating                    3025 non-null   float64
 3   Number of Reviews         3025 non-null   int64  
 4   Price                     3025 non-null   float64
 5   Description               3025 non-null   object 
 6   Listening Time (minutes)  3025 non-null   float64
 7   Genre                     3025 non-null   object 
 8   Rank                      3025 non-null   int64  
 9   combined_text             3025 non-null   object 
 10  cleaned_text              3025 non-null   object 
 11  cluster                   3025 non-null   int64  
dtypes: float64(3), int64(3), object(6)
memory usage: 283.7+ KB


# 1) Content based Recommendation

In [10]:
# Step 1: TF-IDF Vectorization on 'cleaned_text'

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['cleaned_text'])

In [ ]:
# save TF-IDF for deployment

import joblib

joblib.dump(tfidf, './files/tfidf_vectorizer.pkl')

['./files/tfidf_vectorizer.pkl']

In [11]:
# Step 2: Compute Cosine Similarity

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [21]:
# save cosine similarity for deployment

import joblib

joblib.dump(cosine_sim, './files/cosine_similarity_matrix.pkl')

['./files/cosine_similarity_matrix.pkl']

In [ ]:
# Step 3: Create a mapping from book name to index
book_indices = pd.Series(df.index, index=df['Book Name']).drop_duplicates()

In [22]:
# Recommendation function
def get_content_based_recommendations(title, top_n=5):
    idx = book_indices.get(title)
    if idx is None:
        return "Book not found."
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    book_indices_top = [i[0] for i in sim_scores]
    return df[['Book Name', 'Author', 'Genre']].iloc[book_indices_top]

In [23]:
recommendations = get_content_based_recommendations("Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now")
recommendations

,Book Name,Author,Genre
1137,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,Social Media
2729,One Million Followers: How I Built a Massive S...,Brendan Kane,Religion & Philosophy
1657,The Simulation Hypothesis: An MIT Computer Sci...,Rizwan Virk,Cybernetics
839,Social Media Marketing Workbook 2020,Jason McDonald PhD,Social Media
1387,Social Media Marketing 2020 Mastery 4 Books Bu...,Brandon J. Artley,Compulsive Disorders


# 2) Clusterin-Based Recommendation

In [24]:
# Create mapping of book names to their cluster
cluster_map = df.set_index('Book Name')['cluster'].to_dict()

In [25]:
# Define recommendation function
def get_cluster_based_recommendations(book_title, top_n=5):
    if book_title not in cluster_map:
        return "Book not found."
    
    cluster_id = cluster_map[book_title]
    cluster_books = df[df['cluster'] == cluster_id]
    
    # Exclude the input book from recommendations
    recommendations = cluster_books[cluster_books['Book Name'] != book_title]
    
    return recommendations[['Book Name', 'Author', 'Genre']].head(top_n)

In [26]:
book_title = "Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now"
get_cluster_based_recommendations(book_title)

,Book Name,Author,Genre
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,Personal Success
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,Literary Essays
6,Sapiens,Yuval Noah Harari,Anthropology
7,The Intelligent Investor Rev Ed.,Benjamin Graham,Personal Finance
8,Rich Dad Poor Dad: What the Rich Teach Their K...,Robert T. Kiyosaki,Personal Finance


# 3) Hybrid based recommendation

In [27]:
def get_hybrid_recommendations(title, top_n=5):
    idx = book_indices.get(title)
    if idx is None:
        return "Book not found."
    
    # Step 1: Content-based similarity
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:]  # Skip self

    # Step 2: Fetch books from same cluster
    input_cluster = df.loc[idx, 'cluster']

    # Step 3: Score boost based on cluster match and rating
    recommendations = []
    for i, sim in sim_scores:
        cluster_score = 1.2 if df.loc[i, 'cluster'] == input_cluster else 1.0
        rating_score = df.loc[i, 'Rating']
        final_score = sim * cluster_score * rating_score
        recommendations.append((i, final_score))

    # Step 4: Sort and return top N
    top_recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)[:top_n]
    top_indices = [i[0] for i in top_recommendations]
    return df[['Book Name', 'Author', 'Genre', 'Rating']].iloc[top_indices]

In [28]:
get_hybrid_recommendations("Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now")

,Book Name,Author,Genre,Rating
1137,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,Social Media,4.4
839,Social Media Marketing Workbook 2020,Jason McDonald PhD,Social Media,4.5
1657,The Simulation Hypothesis: An MIT Computer Sci...,Rizwan Virk,Cybernetics,4.4
2729,One Million Followers: How I Built a Massive S...,Brendan Kane,Religion & Philosophy,4.3
2307,Built to Serve: Find Your Purpose and Become t...,Evan Carmichael,Fiction Sagas,4.8
